In [145]:
from pathlib import Path
import pandas as pd

In [146]:
# Configuração dos caminhos do projeto

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_FILE = RAW_DIR / "WineQT.csv"
PROCESSED_FILE = PROCESSED_DIR / "wine_quality_clean.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [147]:
# Carregamento dos dados brutos

wine_df = pd.read_csv(RAW_FILE)

print(f"Dimensões originais: {wine_df.shape[0]} linhas e {wine_df.shape[1]} colunas")

wine_df.head()

Dimensões originais: 1143 linhas e 13 colunas


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [148]:
# Visualização inicial da estrutura dos dados

wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


In [149]:
# verificação de valores nulos
print("Contagem de valores nulos por coluna:")
display(wine_df.isnull().sum())

Contagem de valores nulos por coluna:


fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
Id                      0
dtype: int64

In [150]:
wine_df.duplicated().sum()

np.int64(0)

In [151]:
# remoção de duplicatas
features_para_checar = wine_df.drop(columns=["Id"], errors="ignore").columns
duplicated_count = wine_df.duplicated(subset=features_para_checar).sum()

print(f"{duplicated_count} registros duplicados reais encontrados.")

125 registros duplicados reais encontrados.


In [152]:
# Visualizando todos os registros envolvidos em duplicidade

duplicados = wine_df[
    wine_df.duplicated(subset=features_para_checar, keep=False)
].sort_values(by=list(features_para_checar))

print(f"{duplicados.shape[0]} linhas fazem parte de grupos duplicados.")

display(duplicados)

239 linhas fazem parte de grupos duplicados.


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
96,5.2,0.340,0.00,1.8,0.050,27.0,63.0,0.99160,3.68,0.79,14.0,6,142
98,5.2,0.340,0.00,1.8,0.050,27.0,63.0,0.99160,3.68,0.79,14.0,6,144
949,6.0,0.500,0.00,1.4,0.057,15.0,26.0,0.99448,3.36,0.45,9.5,5,1336
950,6.0,0.500,0.00,1.4,0.057,15.0,26.0,0.99448,3.36,0.45,9.5,5,1337
951,6.0,0.500,0.00,1.4,0.057,15.0,26.0,0.99448,3.36,0.45,9.5,5,1338
...,...,...,...,...,...,...,...,...,...,...,...,...,...
260,12.8,0.615,0.66,5.8,0.083,7.0,42.0,1.00220,3.07,0.73,10.0,7,366
400,13.0,0.470,0.49,4.3,0.085,6.0,47.0,1.00210,3.30,0.68,12.7,6,559
404,13.0,0.470,0.49,4.3,0.085,6.0,47.0,1.00210,3.30,0.68,12.7,6,564
170,15.0,0.210,0.44,2.2,0.075,10.0,24.0,1.00005,3.07,0.84,9.2,7,243


In [153]:
if duplicated_count > 0:
    wine_df = wine_df.drop_duplicates(subset=features_para_checar)
    print(f"{duplicated_count} registros duplicados reais removidos para governança do pipeline.")

125 registros duplicados reais removidos para governança do pipeline.


In [154]:
# Criação da variável alvo binária
# 1 = vinho de alta qualidade, nota >= 7
# 0 = vinho de média/baixa qualidade, nota < 7

def classificar_qualidade_binaria(nota):
    """
    Essa é uma função que serve para classificar o vinho em alta qualidade
    ou média/baixa qualidade. Consideramos que vinhos com nota maior ou igual
    a 7 são vinhos de alta qualidade.

    Retorna
    ----------
    1 : Inteiro
      Retorna 1 para os casos em que o vinho seja de alta qualidade.
    0 : Inteiro
      Retorna 0 para os casos em que o vinho seja de média/baixa qualidade.
    """
    if nota >= 7:
        return 1
    return 0


wine_df_clean["quality_binary"] = wine_df_clean["quality"].apply(classificar_qualidade_binaria)

wine_df_clean[["quality", "quality_binary"]].head()

,quality,quality_binary
0,5,0
1,5,0
2,5,0
3,6,0
5,5,0


In [155]:
# Validação da variável alvo criada

distribuicao_alvo = wine_df_clean["quality_binary"].value_counts().sort_index()

print("Distribuição da variável alvo binária:")
display(distribuicao_alvo)

Distribuição da variável alvo binária:


quality_binary
0    881
1    137
Name: count, dtype: int64

In [156]:
# Percentual da variável alvo

percentual_alvo = (wine_df_clean["quality_binary"].value_counts(normalize=True).sort_index() * 100).round(2)

print("Percentual da variável alvo binária:")
display(percentual_alvo)

Percentual da variável alvo binária:


quality_binary
0    86.54
1    13.46
Name: proportion, dtype: float64

In [157]:
# Remover Id da base limpa
wine_df_clean = wine_df_clean.drop(columns=["Id"], errors="ignore")

In [158]:
# Verificação final da base limpa

print(f"Dimensões finais da base limpa: {wine_df_clean.shape[0]} linhas e {wine_df_clean.shape[1]} colunas")

wine_df_clean.head()

Dimensões finais da base limpa: 1018 linhas e 13 colunas


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,quality_binary
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0
5,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5,0


In [159]:
# Salvando a base limpa em data/processed

wine_df_clean.to_csv(PROCESSED_FILE, index=False)

print(f"Base limpa salva em: {PROCESSED_FILE}")

Base limpa salva em: /Users/assis/Documents/Jacque/FIAP/FASE02/projeto/Tech-Challenge-FIAP-Fase-2---Wine-Quality/data/processed/wine_quality_clean.csv
